<a href="https://colab.research.google.com/github/igoldin670/BlackJack-RL/blob/main/BlackJack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Blackjack Simulator

This notebook implements a simple Blackjack simulator that can later be used for Q-learning.

The player starts with two cards. The player can either:

- `"hit"`: draw another card
- `"stand"`: stop and let the dealer play

The game returns one of four outcomes:

- `"in_play"`: the game continues
- `"bust"`: the player went over 21
- `"win"`: the player wins
- `"lose"`: the player loses

In [ ]:
import random

## Create and Shuffle the Deck

We use a global deck variable as the source of randomness for the simulator.

Face cards are represented as `10`.

Aces are represented as `"A"` because they can count as either 1 or 11.

In [ ]:
deck = []


def create_deck():
    return [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, "A"] * 4


def shuffle_deck():
    global deck
    deck = create_deck()
    random.shuffle(deck)


def draw_card():
    global deck

    if len(deck) == 0:
        shuffle_deck()

    return deck.pop()

## Update a Blackjack Hand

The state of a hand is represented as:

```python
(total, usable_ace)
```

`total` is the current value of the hand.

`usable_ace` is True if one ace is currently being counted as 11.

In [ ]:
def add_card_to_state(total, usable_ace, card):
    if card == "A":
      if total + 11 <= 21:
        total += 11
        usable_ace = True
      else:
        total += 1
    else:
      total += card

    if total > 21 and usable_ace:
        total -= 10
        usable_ace = False

    return total, usable_ace

## Set Up a New Game

The `setup_game()` function starts a new game.

It:

1. Shuffles a new deck.
2. Deals two cards to the player.
3. Returns the player's initial state.

The state is:

```python
(player_total, usable_ace)
```

In [ ]:
def setup_game():
    shuffle_deck()

    player_total = 0
    usable_ace = False

    for _ in range(2):
        card = draw_card()
        player_total, usable_ace = add_card_to_state(player_total, usable_ace, card)

    return player_total, usable_ace

## Dealer Turn

The dealer gets two cards.

Then the dealer keeps hitting until their hand value is at least 17.

If the dealer goes over 21, the dealer busts.

In [ ]:
def dealer_play():
    dealer_total = 0
    usable_ace = False

    for _ in range(2):
        card = draw_card()
        dealer_total, usable_ace = add_card_to_state(dealer_total, usable_ace, card)

    while dealer_total < 17:
        card = draw_card()
        dealer_total, usable_ace = add_card_to_state(dealer_total, usable_ace, card)

    return dealer_total

## Transition Function

The `step()` function is the main transition function.

It takes:

```python
state
action
```

where:

```python
state = (player_total, usable_ace)
```

and action is either:

`"hit"`
or
`"stand"`

The function returns:

next_state, outcome

The outcome can be:

`"in_play"`
`"bust"`
`"win"`
`"lose"`

In [ ]:
def step(state, action):
    player_total, usable_ace = state

    if action == "hit":
        card = draw_card()
        player_total, usable_ace = add_card_to_state(player_total, usable_ace, card)

        if player_total > 21:
            return None, "bust"

        return (player_total, usable_ace), "in_play"

    elif action == "stand":
        dealer_total = dealer_play()

        if dealer_total > 21:
            return state, "win"
        elif player_total > dealer_total:
            return state, "win"
        else:
            return state, "lose"

    else:
        raise ValueError("Invalid action: {}".format(action))

## Test One Game

This cell tests one simple game.

The player starts with two cards, hits once, and if the game is still going, stands.

In [ ]:
state = setup_game()
print("Initial player state:", state)

state, outcome = step(state, "hit")
print("After hit:", state, outcome)

if outcome == "in_play":
    state, outcome = step(state, "stand")
    print("After stand:", state, outcome)

Initial player state: (17, False)
After hit: None bust


## Q-Learning Format

For Q-learning, it is often better to return:

```python
next_state, reward, done
```

where:

- win gives reward `+1`
- lose gives reward `-1`
- bust gives reward `-1`
- in play gives reward `0`

The `done` variable is `True` when the game is over.

In [ ]:
def step_q_learning(state, action):
    next_state, outcome = step(state, action)

    if outcome == "win":
        reward = 1
        done = True
    elif outcome == "lose":
        reward = -1
        done = True
    elif outcome == "bust":
        reward = -1
        done = True
    else:
        reward = 0
        done = False

    return next_state, reward, done

In [ ]:
state = setup_game()
print("Initial state:", state)

next_state, reward, done = step_q_learning(state, "hit")

print("Next state:", next_state)
print("Reward:", reward)
print("Done:", done)

Initial state: (13, False)
Next state: None
Reward: -1
Done: True


# Q-Learning for Simple Blackjack

Now we build a Q-learner for the Blackjack simulator.

The Q-table stores the value of each action in each state.

A state looks like:

```python
(player_total, usable_ace)
```

An action is either:

`"hit"`
or
`"stand"`

So one Q-table entry looks like:

```python
Q[(18, False)]["hit"]
Q[(18, False)]["stand"]
```

The value estimates how good that action is from that state.

## Set Up the Q-Table

We use a pandas DataFrame for the Q-table.

Each state has two possible actions:

- `"hit"`
- `"stand"`

At the beginning, all Q-values are set to `0.0`.

In [ ]:
import numpy as np
import pandas as pd

actions = ["hit", "stand"]

player_totals = range(4, 22)
usable_ace_values = [False, True]

state_index = pd.MultiIndex.from_product(
    [player_totals, usable_ace_values],
    names=["player_total", "usable_ace"]
)

Q = pd.DataFrame(
    np.zeros((len(state_index), len(actions))),
    index=state_index,
    columns=actions
)

Q.head(10)

hit  stand
player_total usable_ace            
4            False       0.0    0.0
             True        0.0    0.0
5            False       0.0    0.0
             True        0.0    0.0
6            False       0.0    0.0
             True        0.0    0.0
7            False       0.0    0.0
             True        0.0    0.0
8            False       0.0    0.0
             True        0.0    0.0

## Q-Learning Update Rule

The Q-learning formula is:

```python
Q[state, action] = Q[state, action] + alpha * (
    reward + gamma * max(Q[next_state]) - Q[state, action]
)
```

For this assignment:

`gamma = 1`

because games end quickly, so we do not discount future rewards.


In [ ]:
def update_q_value(Q, state, action, reward, next_state, done, alpha = 0.1, gamma = 1):
    current_q = Q.loc[state, action]

    if done:
      target = reward

    else:
        best_next_q = Q.loc[next_state].max()
        target = reward + gamma * best_next_q

    Q.loc[state, action] = current_q + alpha * (target - current_q)

## Exploration Function

This function plays many blackjack games.

The player chooses actions randomly.

After each action, the Q-table is updated.

In [ ]:
def explore_blackjack(Q, num_games = 10000, alpha = 0.1, gamma = 1):
    results = {
        "win":0,
        "lose":0,
        "bust":0
    }

    for game in range(num_games):
        state = setup_game()
        done = False

        while not done:
            action = random.choice(actions)
            next_state, reward, done = step_q_learning(state, action)
            update_q_value(Q, state, action, reward, next_state, done, alpha, gamma)

            if done:
              if reward == 1:
                results["win"] += 1
              elif next_state is None:
                results["bust"] += 1
              else:
                results["lose"] += 1
            else:
              state = next_state

    return Q, results

## Run Q-Learning

Now we run the exploration function for 10,000 games.

In [ ]:
Q, results = explore_blackjack(
    Q=Q,
    num_games=10000,
    alpha=0.1,
    gamma=1
)

results

{'win': 2801, 'lose': 4040, 'bust': 3159}

In [ ]:
Q

hit     stand
player_total usable_ace                    
4            False      -0.130763 -0.251186
             True        0.000000  0.000000
5            False      -0.249217 -0.590206
             True        0.000000  0.000000
6            False      -0.233112 -0.647659
             True        0.000000  0.000000
7            False      -0.233118 -0.493437
             True        0.000000  0.000000
8            False      -0.096121 -0.267157
             True        0.000000  0.000000
9            False       0.056291 -0.409638
             True        0.000000  0.000000
10           False       0.089899 -0.564032
             True        0.000000  0.000000
11           False       0.065138 -0.659875
             True        0.000000  0.000000
12           False      -0.440090 -0.469198
             True       -0.006155 -0.392606
13           False      -0.448721 -0.321414
             True       -0.101596 -0.193344
14           False      -0.393350 -0.453750
             True       -0.042677 -0.801821
15           False      -0.638799 -0.866074
             True       -0.232810 -0.404050
16           False      -0.452207 -0.668196
             True       -0.238060 -0.348070
17           False      -0.590737 -0.370527
             True       -0.120989 -0.502391
18           False      -0.706458 -0.143596
             True        0.264900  0.173026
19           False      -0.929405  0.381940
             True       -0.085530  0.630577
20           False      -0.767596  0.307031
             True        0.137721  0.410318
21           False      -1.000000  0.797513
             True        0.311956  0.947167

## Best Learned Action for Each State

This shows whether the learned policy prefers `"hit"` or `"stand"` for each state.

In [ ]:
policy = Q.idxmax(axis=1).to_frame(name="best_action")

policy.head(20)

best_action
player_total usable_ace            
4            False              hit
             True               hit
5            False              hit
             True               hit
6            False              hit
             True               hit
7            False              hit
             True               hit
8            False              hit
             True               hit
9            False              hit
             True               hit
10           False              hit
             True               hit
11           False              hit
             True               hit
12           False              hit
             True               hit
13           False            stand
             True               hit

In [ ]:
policy

best_action
player_total usable_ace            
4            False              hit
             True               hit
5            False              hit
             True               hit
6            False              hit
             True               hit
7            False              hit
             True               hit
8            False              hit
             True               hit
9            False              hit
             True               hit
10           False              hit
             True               hit
11           False              hit
             True               hit
12           False              hit
             True               hit
13           False            stand
             True               hit
14           False              hit
             True               hit
15           False              hit
             True               hit
16           False              hit
             True               hit
17           False            stand
             True               hit
18           False            stand
             True               hit
19           False            stand
             True             stand
20           False            stand
             True             stand
21           False            stand
             True             stand

## Combine Q-Values and Best Action

This table shows the Q-value for hitting, the Q-value for standing, and the best action.

In [ ]:
Q_with_policy = Q.copy()
Q_with_policy["best_action"] = Q.idxmax(axis=1)

Q_with_policy

hit     stand best_action
player_total usable_ace                                
4            False      -0.130763 -0.251186         hit
             True        0.000000  0.000000         hit
5            False      -0.249217 -0.590206         hit
             True        0.000000  0.000000         hit
6            False      -0.233112 -0.647659         hit
             True        0.000000  0.000000         hit
7            False      -0.233118 -0.493437         hit
             True        0.000000  0.000000         hit
8            False      -0.096121 -0.267157         hit
             True        0.000000  0.000000         hit
9            False       0.056291 -0.409638         hit
             True        0.000000  0.000000         hit
10           False       0.089899 -0.564032         hit
             True        0.000000  0.000000         hit
11           False       0.065138 -0.659875         hit
             True        0.000000  0.000000         hit
12           False      -0.440090 -0.469198         hit
             True       -0.006155 -0.392606         hit
13           False      -0.448721 -0.321414       stand
             True       -0.101596 -0.193344         hit
14           False      -0.393350 -0.453750         hit
             True       -0.042677 -0.801821         hit
15           False      -0.638799 -0.866074         hit
             True       -0.232810 -0.404050         hit
16           False      -0.452207 -0.668196         hit
             True       -0.238060 -0.348070         hit
17           False      -0.590737 -0.370527       stand
             True       -0.120989 -0.502391         hit
18           False      -0.706458 -0.143596       stand
             True        0.264900  0.173026         hit
19           False      -0.929405  0.381940       stand
             True       -0.085530  0.630577       stand
20           False      -0.767596  0.307031       stand
             True        0.137721  0.410318       stand
21           False      -1.000000  0.797513       stand
             True        0.311956  0.947167       stand

Some states are impossible

Rows like this are impossible:
```python
4, usable_ace=True
5, usable_ace=True
6, usable_ace=True
```

A usable ace means an ace is counted as 11, so the smallest usable-ace total is usually 12, from A + A.

This is not a serious issue. It just means some rows stay at 0.0.

In [ ]:
Q_with_policy = Q.copy()
Q_with_policy["best_action"] = Q.idxmax(axis=1)
Q_with_policy["best_q_value"] = Q.max(axis=1)

Q_with_policy[["hit", "stand", "best_action", "best_q_value"]]

hit     stand best_action  best_q_value
player_total usable_ace                                              
4            False      -0.130763 -0.251186         hit     -0.130763
             True        0.000000  0.000000         hit      0.000000
5            False      -0.249217 -0.590206         hit     -0.249217
             True        0.000000  0.000000         hit      0.000000
6            False      -0.233112 -0.647659         hit     -0.233112
             True        0.000000  0.000000         hit      0.000000
7            False      -0.233118 -0.493437         hit     -0.233118
             True        0.000000  0.000000         hit      0.000000
8            False      -0.096121 -0.267157         hit     -0.096121
             True        0.000000  0.000000         hit      0.000000
9            False       0.056291 -0.409638         hit      0.056291
             True        0.000000  0.000000         hit      0.000000
10           False       0.089899 -0.564032         hit      0.089899
             True        0.000000  0.000000         hit      0.000000
11           False       0.065138 -0.659875         hit      0.065138
             True        0.000000  0.000000         hit      0.000000
12           False      -0.440090 -0.469198         hit     -0.440090
             True       -0.006155 -0.392606         hit     -0.006155
13           False      -0.448721 -0.321414       stand     -0.321414
             True       -0.101596 -0.193344         hit     -0.101596
14           False      -0.393350 -0.453750         hit     -0.393350
             True       -0.042677 -0.801821         hit     -0.042677
15           False      -0.638799 -0.866074         hit     -0.638799
             True       -0.232810 -0.404050         hit     -0.232810
16           False      -0.452207 -0.668196         hit     -0.452207
             True       -0.238060 -0.348070         hit     -0.238060
17           False      -0.590737 -0.370527       stand     -0.370527
             True       -0.120989 -0.502391         hit     -0.120989
18           False      -0.706458 -0.143596       stand     -0.143596
             True        0.264900  0.173026         hit      0.264900
19           False      -0.929405  0.381940       stand      0.381940
             True       -0.085530  0.630577       stand      0.630577
20           False      -0.767596  0.307031       stand      0.307031
             True        0.137721  0.410318       stand      0.410318
21           False      -1.000000  0.797513       stand      0.797513
             True        0.311956  0.947167       stand      0.947167